## Comparing Movie Recommendation Ranking methods

Here we experiment whether ranking movies based on difference between a user's rating habits/predicted ratings and a movie's average rating. We question whether the relationship between these two variables will produce more personalized reccomendation over ranking movies by their predicted ratings.

So here we compare: Standard Ranking and Difference-based ranking where Standard ranking ranks movies from their highest to lowest predicted rating and Difference-based ranking is where movies are ranked by the user's rating habits/predicted ratings compares to the movie's average rating. Then we examine the number of ratings ech recommended movie has recieved, this is to determine which method favors more popular movies.

In simple terms: 
Standard Ranking: recommends movies based on how highly the model predicts a user will rate that movie.
Difference-based ranking: recommends movies the model predicts a user will enjoy more than the average viewer does.


In [1]:

import pandas as pd
from pathlib import Path
from surprise import Dataset, Reader, SVD

# Load the Letterboxd dataset
path = Path("../reviews_filtered.csv")

if not path.exists():
    path = Path("reviews_filtered.csv")

data = pd.read_csv(path)

print("Total ratings:", len(data))
print("Unique movies:", data["movie_id"].nunique())
print("Unique users:", data["user_id"].nunique())

# Load ratings into Surprise
reader = Reader(rating_scale=(1, 10))

dataset = Dataset.load_from_df(
    data[["user_id", "movie_id", "rating"]],
    reader
)

# Train SVD model
model = SVD(random_state=42)
model.fit(dataset.build_full_trainset())


Total ratings: 855434
Unique movies: 5937
Unique users: 2516


In [2]:
# Select an existing user rating history
user_counts = data["user_id"].value_counts()
user_id = user_counts.index[min(100, len(user_counts) - 1)]

# Find movies user has already rated
rated_movies = set(
    data.loc[data["user_id"] == user_id, "movie_id"]
)

# Calculate average ratings and rating counts for each movie
movie_stats = (
    data.groupby("movie_id")
    .agg(
        name=("name", "first"),
        avg_rating=("rating", "mean"),
        rating_count=("rating", "size"))
    .reset_index()
)

# Keep movies user hasn't rated and discard already rated movies
candidates = movie_stats[ ~movie_stats["movie_id"].isin(rated_movies)].copy()

# Predict the user's rating for each unrated movie
candidates["predicted_rating"] = [
    model.predict(user_id, movie_id).est
    for movie_id in candidates["movie_id"]
]

# Calculate the difference between predicted and average rating
candidates["difference"] = (
    candidates["predicted_rating"] - candidates["avg_rating"]
)

#showing some comparison
print("Selected user:", user_id)
print("Movies already rated:", len(rated_movies))
print("Movies available for recommendations:", len(candidates))

display(candidates.head(10))


Selected user: 499263
Movies already rated: 1203
Movies available for recommendations: 4734


,movie_id,name,avg_rating,rating_count,predicted_rating,difference
0,247,take-shelter,7.918605,86,7.836554,-0.082050
4,281,47-ronin-2013,4.029412,34,3.501098,-0.528313
5,282,extremely-loud-incredibly-close,4.763158,38,2.438332,-2.324826
6,285,the-great-gatsby-2013,6.092937,269,4.291889,-1.801048
7,331,total-recall-2012,4.717391,46,3.167382,-1.550009
8,603,the-muppets,7.353846,130,5.743165,-1.610681
9,994,jackass-35,7.108108,37,6.424899,-0.683209
10,1081,happy-feet-two,4.341463,41,4.528619,0.187155
13,1610,lemonade-mouth,7.333333,78,4.819393,-2.513940
14,1739,the-woman-in-black-2012,5.541176,85,4.606222,-0.934954


Process in the cell:
1. select a user that has a good history in rating movies
2. remove all the movies they already rated
3. use the SVD model we trained earler to predict how a user would rate the remaining movies
4. Calculate the difference between movies average rating from their predicted rating.

In [3]:
#sorting based on highest recommendation score
# i think these values displayed might indicate that the model is just recommending movies based on their popularity
# as the scale of the predicted ratings is very similar to the average ratings



top_by_prediction = candidates.sort_values(
    "predicted_rating", ascending=False
).head(10)

mid_by_prediction = candidates.sort_values(
    "predicted_rating", ascending=False
).head(candidates.shape[0]//2).tail(10)

bottom_by_prediction = candidates.sort_values(
    "predicted_rating", ascending=True
).head(10)

print("Top 10 recommended movies:")
display(top_by_prediction[
    ["name", "predicted_rating", "avg_rating", "rating_count", "difference"]
])

print("Middle 10 recommended movies:")
display(mid_by_prediction[
    ["name", "predicted_rating", "avg_rating", "rating_count", "difference"]
])

print("Bottom 10 recommended movies:")
display(bottom_by_prediction[
    ["name", "predicted_rating", "avg_rating", "rating_count", "difference"]
])

Top 10 recommended movies:


,name,predicted_rating,avg_rating,rating_count,difference
1564,twin-peaks-fire-walk-with-me,10.000000,8.624060,133,1.375940
379,come-and-see,10.000000,9.377358,106,0.622642
5321,the-beast-2023-1,10.000000,7.832317,328,2.167683
3791,twin-peaks-the-return,10.000000,9.475000,160,0.525000
1712,mulholland-drive,9.805985,8.875536,233,0.930448
2553,inherent-vice,9.755574,7.610619,226,2.144954
484,neon-genesis-evangelion-the-end-of-evangelion,9.751894,8.823529,68,0.928364
1903,sunrise-a-song-of-two-humans,9.694628,8.571429,42,1.123199
2249,lawrence-of-arabia,9.607348,8.565789,76,1.041559
29,citizen-kane,9.577716,8.500000,144,1.077716


Middle 10 recommended movies:


,name,predicted_rating,avg_rating,rating_count,difference
4847,one-life-2023,5.475698,6.781818,110,-1.306120
4382,the-harder-they-fall-2021,5.474608,6.720165,243,-1.245557
3283,my-life-as-a-zucchini,5.471267,8.028571,140,-2.557305
2961,the-founder,5.470635,6.128866,194,-0.658231
2352,seeking-a-friend-for-the-end-of-the-world,5.470167,6.450704,71,-0.980538
5727,anyone-but-you,5.469669,5.942744,1013,-0.473075
4997,ticket-to-paradise-2022,5.469186,5.760116,346,-0.290929
4464,shithouse,5.468433,7.466019,206,-1.997587
2646,macbeth-2015,5.467737,6.680556,72,-1.212818
4203,news-of-the-world,5.466850,6.116279,172,-0.649429


Bottom 10 recommended movies:


,name,predicted_rating,avg_rating,rating_count,difference
4786,after-we-fell,1.0,2.033708,89,-1.033708
5098,the-next-365-days,1.0,1.434783,46,-0.434783
1038,epic-movie,1.0,1.695652,46,-0.695652
4219,the-kissing-booth-2,1.0,2.222749,211,-1.222749
618,disaster-movie,1.0,1.479167,48,-0.479167
3000,nine-lives-2016,1.0,2.536585,41,-1.536585
2750,pixels-2015,1.0,3.472050,161,-2.472050
3646,resident-evil-welcome-to-raccoon-city,1.0,4.744186,215,-3.744186
2718,gods-not-dead,1.0,1.632653,49,-0.632653
5025,the-exorcist-believer,1.0,3.897590,498,-2.897590


In [4]:
from surprise import Dataset, Reader, SVD


def evaluate_fixed_holdouts(
    data,
    holdout_movies_by_user,
    like_threshold=8,
    k=10,
    seed=42,
):
    """
    Hide specified movie ratings per randomly selected sample of users, 
    train SVD on the remaining data,
    and compare predicted-rating and popularity rankings.

    Helper function to the set of selected users. 

    holdout_movies_by_user format:
        {user_id: [movie_id_1, movie_id_2, ...]}
    these movies are randomized for each user. should we keep the list of movies the same across
    all users? I'm thinking that may introduce bias towards the average rating of the movie, 
    and will cause the model to overfit. 

    Returns:
        summary: average precision/recall by ranking method
        per_user_metrics: metric values per user and method
        top_lists: top-k recommendations for each user and method
        heldout_movies: selected holdouts with predictions and training counts
    """
    required = {"user_id", "movie_id", "name", "rating"}
    missing = required - set(data.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    if not isinstance(k, int) or k <= 0:
        raise ValueError("k must be a positive integer.")

    if data.duplicated(["user_id", "movie_id"]).any():
        raise ValueError(
            "Duplicate user/movie rows found. Resolve these before evaluating."
        )

    if not holdout_movies_by_user:
        raise ValueError("Provide at least one user and holdout movie.")

    existing_pairs = set(zip(data["user_id"], data["movie_id"]))
    requested_pairs = set()

    for user_id, movie_ids in holdout_movies_by_user.items():
        if len(movie_ids) != len(set(movie_ids)):
            raise ValueError(f"Duplicate holdout movie IDs for user {user_id}.")

        requested_pairs.update((user_id, movie_id) for movie_id in movie_ids)

    missing_pairs = requested_pairs - existing_pairs
    if missing_pairs:
        raise ValueError(
            "Some requested user/movie pairs are not in the dataset. "
            f"Examples: {list(missing_pairs)[:5]}"
        )

    if not requested_pairs:
        raise ValueError("No holdout movie pairs were provided.")

    # Remove only the specified user/movie ratings from training.
    holdout_mask = [
        (user_id, movie_id) in requested_pairs
        for user_id, movie_id in zip(data["user_id"], data["movie_id"])
    ]
    test_data = data.loc[holdout_mask].copy()
    train_data = data.loc[[not held_out for held_out in holdout_mask]].copy()

    for user_id in holdout_movies_by_user:
        if not (train_data["user_id"] == user_id).any():
            raise ValueError(
                f"User {user_id} has no ratings left for model training."
            )

    # Train SVD using all rows remaining in train_data.
    reader = Reader(rating_scale=(1, 10))
    surprise_data = Dataset.load_from_df(
        train_data[["user_id", "movie_id", "rating"]],
        reader,
    )
    trainset = surprise_data.build_full_trainset()

    model = SVD(random_state=seed)
    model.fit(trainset)

    # Keep movie names from the full catalog, but calculate popularity
    # using training ratings only.
    movie_catalog = data[["movie_id", "name"]].drop_duplicates("movie_id")
    training_movie_stats = (
        train_data.groupby("movie_id")["rating"]
        .agg(avg_rating="mean", training_rating_count="size")
        .reset_index()
    )
    training_mean = train_data["rating"].mean()

    metric_rows = []
    top_list_rows = []
    heldout_rows = []

    for user_id in holdout_movies_by_user:
        user_test = test_data[test_data["user_id"] == user_id]

        relevant_movie_ids = set(
            user_test.loc[
                user_test["rating"] >= like_threshold,
                "movie_id",
            ]
        )

        # A candidate is any movie the user did not rate in training.
        seen_in_training = set(
            train_data.loc[
                train_data["user_id"] == user_id,
                "movie_id",
            ]
        )
        candidates = movie_catalog[
            ~movie_catalog["movie_id"].isin(seen_in_training)
        ].copy()

        candidates["predicted_rating"] = [
            model.predict(user_id, movie_id).est
            for movie_id in candidates["movie_id"]
        ]

        candidates = candidates.merge(
            training_movie_stats,
            on="movie_id",
            how="left",
        )
        candidates["avg_rating"] = candidates["avg_rating"].fillna(training_mean)
        candidates["training_rating_count"] = (
            candidates["training_rating_count"].fillna(0).astype(int)
        )

        # Save every held-out movie with its prediction and training count.
        user_heldout_details = candidates.loc[
            candidates["movie_id"].isin(user_test["movie_id"]),
            [
                "movie_id",
                "name",
                "predicted_rating",
                "avg_rating",
                "training_rating_count",
            ],
        ].merge(
            user_test[["movie_id", "rating"]].rename(
                columns={"rating": "heldout_rating"}
            ),
            on="movie_id",
            how="left",
        )

        user_heldout_details.insert(0, "user_id", user_id)
        user_heldout_details["is_relevant"] = (
            user_heldout_details["heldout_rating"] >= like_threshold
        )
        heldout_rows.append(user_heldout_details)

        # Keep the two ranking methods unchanged.
        rankings = {
            "SVD predicted rating": candidates.sort_values(
                ["predicted_rating", "movie_id"],
                ascending=[False, True],
            ),
            "Popularity": candidates.sort_values(
                ["training_rating_count", "movie_id"],
                ascending=[False, True],
            ),
        }

        heldout_rating_by_movie = user_test.set_index("movie_id")["rating"]

        for method, ranking in rankings.items():
            top_k = ranking.head(k).copy()
            recommended_ids = set(top_k["movie_id"])

            # Save the top-k recommendations for this user and method.
            top_k["heldout_rating"] = top_k["movie_id"].map(
                heldout_rating_by_movie
            )
            top_k["has_heldout_rating"] = top_k["heldout_rating"].notna()
            top_k["is_relevant"] = (
                top_k["heldout_rating"].ge(like_threshold).fillna(False)
            )
            top_k.insert(0, "rank", range(1, len(top_k) + 1))
            top_k.insert(0, "method", method)
            top_k.insert(0, "user_id", user_id)
            top_list_rows.append(top_k)

            # Precision/recall can be computed if this user has relevant
            # held-out movies at the chosen threshold.
            if relevant_movie_ids:
                hits = len(recommended_ids & relevant_movie_ids)

                metric_rows.append({
                    "user_id": user_id,
                    "method": method,
                    "hits_in_top_k": hits,
                    "relevant_heldout_movies": len(relevant_movie_ids),
                    "precision_at_k": hits / len(top_k),
                    "recall_at_k": hits / len(relevant_movie_ids),
                })

    per_user_metrics = pd.DataFrame(metric_rows)
    top_lists = pd.concat(top_list_rows, ignore_index=True)
    heldout_movies = pd.concat(heldout_rows, ignore_index=True)

    if per_user_metrics.empty:
        summary = pd.DataFrame(
            columns=[
                "precision_at_k",
                "recall_at_k",
                "users_evaluated",
            ]
        )
    else:
        summary = (
            per_user_metrics
            .groupby("method")
            .agg(
                precision_at_k=("precision_at_k", "mean"),
                recall_at_k=("recall_at_k", "mean"),
                users_evaluated=("user_id", "nunique"),
            )
            .round(3)
        )

    return summary, per_user_metrics, top_lists, heldout_movies

In [5]:
import numpy as np

NUM_TEST_USERS = 40 #how many users to select for testing
MIN_RATINGS = 50 # minimum number of ratings a user must have (in order to be properly split)
HOLDOUT_FRACTION = 0.20 #percentage of movie ratings to be exempt
SEED = 42

ratings_per_user = data.groupby("user_id").size()
eligible_users = ratings_per_user[
    ratings_per_user >= MIN_RATINGS
].index.to_numpy()

if len(eligible_users) < NUM_TEST_USERS:
    raise ValueError(
        f"Only {len(eligible_users)} users have at least "
        f"{MIN_RATINGS} ratings."
    )

# Select a reproducible set of users.
user_rng = np.random.default_rng(SEED)
test_user_ids = user_rng.choice(
    eligible_users,
    size=NUM_TEST_USERS,
    replace=False,
).tolist()

# For each selected user, select movies they actually rated.
holdout_rng = np.random.default_rng(SEED + 1)
holdout_movies_by_user = {}

for user_id in test_user_ids:
    user_movie_ids = data.loc[
        data["user_id"] == user_id,
        "movie_id",
    ].to_numpy()

    n_holdout = max(
        1,
        round(len(user_movie_ids) * HOLDOUT_FRACTION),
    )

    holdout_movies_by_user[user_id] = holdout_rng.choice(
        user_movie_ids,
        size=n_holdout,
        replace=False,
    ).tolist()

print(f"Selected {len(test_user_ids)} test users.")
print("Example holdouts:", list(holdout_movies_by_user.items())[:2])

Selected 40 test users.
Example holdouts: [(771936, [575043, 272503, 48044, 251872, 64393, 451512, 335384, 354251, 523229, 82488, 558956, 715856, 51896, 895130, 551275, 794063, 952557, 69142, 362977, 937215, 37550, 923207, 307239, 16527, 59281, 93676, 268192, 44580, 131984, 354541, 48140, 401687, 774923, 251937, 575258, 444424, 7846, 361725, 296341, 95113, 309555, 850459, 16525, 556785, 73371, 878341, 387238, 125976, 338883, 695629, 546348, 459585, 16530, 589093, 178123, 777185, 526697, 495969, 370178, 207224, 705221, 174928, 459564, 395495, 788966, 284629, 62780, 527841, 789082, 389270, 617443, 52512, 8084, 503890, 488575, 431888, 50955, 496592, 472465, 86991, 268947, 494166, 475370, 47994, 518787, 51614, 556553, 755564, 793883, 946607, 436873, 468929, 270477, 129662, 866275, 150170, 41512, 438511, 52513, 50621, 197012, 332434, 47469, 515909, 378525, 686389, 605262, 261866, 691815, 558056, 333310, 27809, 81909, 311120]), (77148, [706064, 703943, 866275, 709530, 427970, 835774, 449442,

information evaluation section. the output is for ALL users so if you don't want that part, **don't run the second cell (it's 7 for me right now).** 

precision and recall table: displays precision and recall for the SVD model and popularity. the hist_in_top_k counts how many top 10 movies were exempt from training AND rated at or above the like threshold. precision is hits/recommended number, recall is hits/user's exempt and above like level movies. 

selected held out movies: all the movies exempt from training for that user. the held out value is the actual rating they gave and predicted_rating is the SVD's estimate. 

top recommendations by method: shows the top 10 candidate movies for each ranking method (popularity, SVD). SVD sorts by predicted rating, popularity sorts by rating count. the held out rating value is only given if it was exempt, otherwise it was used in training. 

In [6]:
# Compare standard and difference-based rankings

# Only recommend movies the user is predicted to enjoy
eligible = candidates[
    candidates["predicted_rating"] >= 7
].copy()

# Method 1: Highest predicted ratings
standard_top10 = eligible.sort_values(
    "predicted_rating", ascending=False
).head(10)

# Method 2: Largest positive differences
difference_top10 = eligible.sort_values(
    "difference", ascending=False
).head(10)

columns = [
    "name",
    "predicted_rating",
    "avg_rating",
    "difference",
    "rating_count"
]

print("STANDARD TOP 10")
display(standard_top10[columns].round(2))

print("DIFFERENCE-BASED TOP 10")
display(difference_top10[columns].round(2))


STANDARD TOP 10


,name,predicted_rating,avg_rating,difference,rating_count
3791,twin-peaks-the-return,10.00,9.48,0.53,160
1564,twin-peaks-fire-walk-with-me,10.00,8.62,1.38,133
5321,the-beast-2023-1,10.00,7.83,2.17,328
379,come-and-see,10.00,9.38,0.62,106
1712,mulholland-drive,9.81,8.88,0.93,233
2553,inherent-vice,9.76,7.61,2.14,226
484,neon-genesis-evangelion-the-end-of-evangelion,9.75,8.82,0.93,68
1903,sunrise-a-song-of-two-humans,9.69,8.57,1.12,42
2249,lawrence-of-arabia,9.61,8.57,1.04,76
29,citizen-kane,9.58,8.50,1.08,144


DIFFERENCE-BASED TOP 10


,name,predicted_rating,avg_rating,difference,rating_count
5321,the-beast-2023-1,10.00,7.83,2.17,328
2553,inherent-vice,9.76,7.61,2.14,226
150,cosmopolis,8.20,6.33,1.87,75
2601,blackhat,7.95,6.48,1.47,123
2566,they-came-together,7.98,6.57,1.41,69
4476,the-northman,9.15,7.77,1.38,1023
1564,twin-peaks-fire-walk-with-me,10.00,8.62,1.38,133
1576,fear-and-loathing-in-las-vegas,8.70,7.33,1.37,97
2373,under-the-skin-2013,8.73,7.42,1.30,257
1665,sex-lies-and-videotape,9.30,8.00,1.30,64


In [7]:

# Compare the two top-10 lists

shared_movies = set(
    standard_top10["movie_id"]
) & set(
    difference_top10["movie_id"]
)

print("Number of movies shared:", len(shared_movies))

print("\nAverage number of movie ratings:")
print("Standard:", round(standard_top10["rating_count"].mean(), 2))
print("Difference:", round(difference_top10["rating_count"].mean(), 2))

print("\nMedian number of movie ratings:")
print("Standard:", standard_top10["rating_count"].median())
print("Difference:", difference_top10["rating_count"].median())


Number of movies shared: 3

Average number of movie ratings:
Standard: 151.6
Difference: 239.5

Median number of movie ratings:
Standard: 138.5
Difference: 128.0


### Observations and Conclusion

From this experiment we see that standard and difference-based ranking methods produced noticeably different movie recommendations for the selected user. We saw 3 out of 10 movies appear in both lists which suggests that the two ranking systems are difference and recommend different movies based on the ranking style.

We also look into the movie's popularity by observing the number of recorded ratings. We saw that standard ranking has an avergae of 151.6 ratings per recommeded movie while difference based ranking had an average of 239.5. The median rating counts were 138.5 and 128 respectively, this combined with the average rating doesn't provide clear evidence that difference based ranking favors less popular movies.

Although difference-based ranking method produced different recommendations, this experiement only analyzed one user, some further evaluation would be necessary to determine if these recommendations are more relevant or accurate than recommendations made with standard ranking.

In [8]:
from IPython.display import display, HTML

summary, per_user_metrics, top_lists, heldout_movies = evaluate_fixed_holdouts(
    data=data,
    holdout_movies_by_user=holdout_movies_by_user,
    like_threshold=8,
    k=10,
    seed=SEED,
)

# Overall comparison of SVD and Popularity.
display(HTML("<h2>Overall precision and recall summary</h2>"))
display(summary.reset_index())

# Rating error uses every held-out rating, not only relevant movies or top-k items.
rating_errors = (
    heldout_movies["predicted_rating"]
    - heldout_movies["heldout_rating"]
)
squared_errors = rating_errors ** 2
user_mse = squared_errors.groupby(heldout_movies["user_id"]).mean()

svd_error_summary = pd.DataFrame([{
    "ratings_evaluated": len(rating_errors),
    "mse": squared_errors.mean(),
    "rmse": np.sqrt(squared_errors.mean()),
    "mean_user_mse": user_mse.mean(),
    "mean_user_rmse": np.sqrt(user_mse).mean(),
}])

display(HTML("<h2>SVD rating error on all held-out ratings</h2>"))
display(svd_error_summary.round(3))

,method,precision_at_k,recall_at_k,users_evaluated
0,Popularity,0.225,0.104,40
1,SVD predicted rating,0.055,0.014,40


,ratings_evaluated,mse,rmse,mean_user_mse,mean_user_rmse
0,3482,2.908,1.705,2.722,1.586


In [9]:
# WARNING: OUTPUT IS REALLY LONG

sections = ["""
<style>
.user-section { margin: 24px 0; padding: 14px; border: 1px solid #bbb; }
.user-section h2 { margin-top: 0; }
.user-section h3 { margin-bottom: 6px; }
table { border-collapse: collapse; margin-bottom: 16px; }
th, td { border: 1px solid #ccc; padding: 4px 8px; text-align: left; }
</style>
"""]

for user_id in holdout_movies_by_user:
    user_metrics = per_user_metrics[
        per_user_metrics["user_id"] == user_id
    ].sort_values("method")

    user_holdouts = heldout_movies[
        heldout_movies["user_id"] == user_id
    ].sort_values("movie_id")

    user_recommendations = top_lists[
        top_lists["user_id"] == user_id
    ].sort_values(["method", "rank"])

    sections.append(f'<section class="user-section"><h2>User {user_id}</h2>')

    sections.append("<h3>Precision and recall</h3>")
    if user_metrics.empty:
        sections.append("<p>No precision/recall scores for this user.</p>")
    else:
        sections.append(
            user_metrics.drop(columns=["user_id"], errors="ignore").to_html(
                index=False,
                na_rep="—",
                border=0,
            )
        )

    sections.append("<h3>Selected held-out movies</h3>")
    if user_holdouts.empty:
        sections.append("<p>No held-out movie rows found for this user.</p>")
    else:
        holdout_columns = [
            column for column in [
                "movie_id",
                "name",
                "predicted_rating",
                "avg_rating",
                "training_rating_count",
                "heldout_rating",
                "is_relevant",
                "meets_like_threshold",
            ]
            if column in user_holdouts.columns
        ]
        sections.append(
            user_holdouts[holdout_columns].to_html(
                index=False,
                na_rep="—",
                border=0,
            )
        )

    sections.append("<h3>Top 10 recommendations by method</h3>")
    if user_recommendations.empty:
        sections.append("<p>No recommendation rows found for this user.</p>")
    else:
        recommendation_columns = [
            column for column in [
                "method",
                "rank",
                "movie_id",
                "name",
                "predicted_rating",
                "avg_rating",
                "training_rating_count",
                "heldout_rating",
                "is_relevant",
                "meets_like_threshold",
            ]
            if column in user_recommendations.columns
        ]
        sections.append(
            user_recommendations[recommendation_columns].to_html(
                index=False,
                na_rep="—",
                border=0,
            )
        )

    sections.append("</section>")

display(HTML("\n".join(sections)))